# 02. DATA CLEANING - LÀM SẠCH VÀ XỬ LÝ DỮ LIỆU

**Mục tiêu:**
1. Overview dữ liệu trước khi làm sạch
2. Xác định và xử lý giá trị thiếu (Missing values)
3. Phát hiện và loại bỏ bản ghi trùng lặp (Duplicates)
4. Xử lý các giá trị ngoại lệ (Outliers)
5. Chuẩn hóa kiểu dữ liệu (Data types)
6. Chuẩn hóa và scale các features (Normalization & Scaling)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import warnings

warnings.filterwarnings('ignore')

print("✅ Các thư viện đã sẵn sàng!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")


## 1. Tải và Khám Phá Dữ Liệu Gốc

Bắt đầu với một overview toàn bộ dữ liệu trước khi làm sạch.

In [ ]:
# Tải dữ liệu đã gộp
# df = pd.read_csv('../data/processed/spotify_merged_with_features.csv') #tien
df = pd.read_csv('../data/processed/spotify_final_imputed.csv')

print("="*80)
print("📊 OVERVIEW DỮ LIỆU GỐC (CHƯA LÀM SẠCH)")
print("="*80)

# 1. Shape của dữ liệu
print(f"\n📈 Shape: {df.shape}")
print(f"   - Số dòng: {df.shape[0]:,}")
print(f"   - Số cột: {df.shape[1]}")

# 2. Các cột dữ liệu
print(f"\n📋 Các cột trong dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

# 3. Kiểu dữ liệu
print(f"\n🔍 Kiểu dữ liệu:")
print(df.dtypes)

# 4. 5 dòng đầu tiên
print(f"\n👀 5 dòng đầu tiên:")
print(df.head())

# 5. Thông tin chi tiết
print(f"\n📊 Thông tin chi tiết:")
df.info()

# 6. Thống kê cơ bản
print(f"\n📈 Thống kê cơ bản (Numerical columns):")
print(df.describe())


In [ ]:
# Reload dữ liệu từ file mới nhất
df = pd.read_csv('../data/processed/spotify_final_imputed.csv')

print("="*80)
print("📊 OVERVIEW DỮ LIỆU GỐC (CHƯA LÀM SẠCH)")
print("="*80)

# 1. Shape của dữ liệu
print(f"\n📈 Shape: {df.shape}")
print(f"   - Số dòng: {df.shape[0]:,}")
print(f"   - Số cột: {df.shape[1]}")

# 2. Các cột dữ liệu
print(f"\n📋 Các cột trong dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

# 3. Kiểu dữ liệu
print(f"\n🔍 Kiểu dữ liệu:")
print(df.dtypes)

# 4. 5 dòng đầu tiên
print(f"\n👀 5 dòng đầu tiên:")
print(df.head())

# 5. Thông tin chi tiết
print(f"\n📊 Thông tin chi tiết:")
df.info()

# 6. Thống kê cơ bản
print(f"\n📈 Thống kê cơ bản (Numerical columns):")
print(df.describe())

In [ ]:
# Overview trực quan các giá trị thiếu
print("\n" + "="*80)
print("❓ PHÂN TÍCH GIÁ TRỊ THIẾU (MISSING VALUES)")
print("="*80)

# Đếm missing values
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_count.index,
    'Missing Count': missing_count.values,
    'Missing Percentage': missing_percent.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Percentage', ascending=False)

if len(missing_df) > 0:
    print(f"\n⚠️ Các cột có giá trị thiếu:")
    print(missing_df.to_string(index=False))
else:
    print(f"\n✅ Không có giá trị thiếu!")

# Visualize missing values
fig, ax = plt.subplots(figsize=(14, 6))
missing_percent_sorted = missing_count[missing_count > 0].sort_values(ascending=False)
if len(missing_percent_sorted) > 0:
    missing_percent_sorted.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Số lượng giá trị thiếu')
    ax.set_title('Missing Values trong Dataset')
    plt.tight_layout()
    plt.show()
else:
    print("Không có visualize vì không có missing values")


## 2. Xử Lý Giá Trị Thiếu (Missing Values)

Đầu tiên, xóa những dòng không có dữ liệu thuộc tính âm học

Kiểm tra các cột có độ khuyết thiếu > 50% -> Xóa cột

In [ ]:
# Giả sử df là DataFrame gốc của bạn
df_clean = df.copy()

print("="*80)
print("🔧 CHIẾN LƯỢC KẾT HỢP: XÓA DÒNG ÂM HỌC THIẾU -> XÓA CỘT THIẾU > 50%")
print("="*80)

print(f"[TRƯỚC KHI XỬ LÝ]")
print(f"   Số dòng: {len(df_clean):,}")
print(f"   Tổng số ô trống (NaN): {df_clean.isnull().sum().sum():,}")

# ---------------------------------------------------------
# BƯỚC 1: XÓA NHỮNG DÒNG KHÔNG CÓ THUỘC TÍNH ÂM HỌC
# ---------------------------------------------------------
# Khai báo các cột âm học quan trọng làm mỏ neo
acoustic_cols = ['danceability', 'energy', 'tempo'] # Bạn có thể thêm 'valence', 'loudness',... vào list này

# Thêm tham số how='any' (mặc định) nghĩa là: Chỉ cần thiếu 1 trong các cột này là xóa nguyên dòng
df_clean = df_clean.dropna(subset=acoustic_cols).reset_index(drop=True)

print(f"[SAU BƯỚC 1: XÓA DÒNG]")
print(f"   Số dòng SẠCH còn lại: {len(df_clean):,}")
print(f"   Tổng số ô trống (NaN) còn lại: {df_clean.isnull().sum().sum():,}")


# ---------------------------------------------------------
# BƯỚC 2: XÓA NHỮNG CỘT CÓ TỶ LỆ THIẾU TRÊN 50%
# ---------------------------------------------------------
cols_to_drop = []

for col in df_clean.columns:
    missing_count = df_clean[col].isnull().sum()
    if missing_count > 0:
        # Tính tỷ lệ % dựa trên số dòng CÒN LẠI sau Bước 1
        missing_percent = (missing_count / len(df_clean)) * 100 
        
        if missing_percent > 50:
            cols_to_drop.append(col)
            print(f"Phát hiện cột '{col}' thiếu {missing_percent:.2f}% -> Chuẩn bị xóa")

# Tiến hành xóa các cột đã lọt vào danh sách
if len(cols_to_drop) > 0:
    df_clean = df_clean.drop(columns=cols_to_drop)
    print(f"[SAU BƯỚC 2: XÓA CỘT] Đã xóa thành công {len(cols_to_drop)} cột.")
else:
    print(f"[SAU BƯỚC 2: XÓA CỘT] Không có cột nào bị khuyết trên 50%.")

print(f"KẾT QUẢ CUỐI CÙNG:")
print(f"   Shape mới: {df_clean.shape}")
print(f"   Tổng số ô trống (NaN) hiện tại: {df_clean.isnull().sum().sum():,}")

# Lưu ý: Bước 3 (fillna) sẽ được thực hiện ở cuối cell 12 (Outliers)

## 3. Phát Hiện và Xóa Bản Ghi Trùng Lặp (Duplicates)

In [ ]:
print("="*80)
print("🔍 PHÁT HIỆN VÀ XÓA BẢN GHI TRÙNG LẶP")
print("="*80)

# Kiểm tra duplicates toàn bộ
full_duplicates = df_clean.duplicated().sum()
print(f"\n🔎 Bản ghi trùng lặp (toàn bộ cột): {full_duplicates}")

# Kiểm tra duplicates theo track_id
if 'track_id' in df_clean.columns:
    track_duplicates = df_clean.duplicated(subset=['track_id']).sum()
    print(f"🔎 Bản ghi trùng lặp (theo track_id): {track_duplicates}")
    
    if track_duplicates > 0:
        print("\n📋 Ví dụ bản ghi trùng:")
        print(df_clean[df_clean.duplicated(subset=['track_id'], keep=False)][['track_id', 'track_name', 'artist_name']].head(10))

# Xóa duplicates
if full_duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"\n✅ Đã xóa {full_duplicates} bản ghi trùng lặp")

print(f"\n📊 Shape sau xóa duplicates: {df_clean.shape}")


## 4. Phát Hiện và Xử Lý Giá Trị Ngoại Lệ (Outliers)

In [ ]:
print("="*80)
print("🎯 PHÁT HIỆN VÀ XỬ LÝ GIÁ TRỊ NGOẠI LỆ (IQR METHOD)")
print("="*80)

# Lấy các cột numerical (audio features)
numeric_features = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Chọn audio features chính để detect outliers
audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness', 
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]
audio_features = [f for f in audio_features if f in numeric_features]

print(f"\n🔍 Phát hiện outliers cho {len(audio_features)} audio features:")

outlier_count_before = len(df_clean)

# Sử dụng IQR method để phát hiện outliers
for col in audio_features:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
    outlier_pct = (len(outliers) / len(df_clean)) * 100
    
    if len(outliers) > 0:
        print(f"\n   • '{col}':")
        print(f"     - Range: [{lower_bound:.2f}, {upper_bound:.2f}]")
        print(f"     - Outliers: {len(outliers)} ({outlier_pct:.2f}%)")

# Xử lý outliers: Thay thế bằng IQR bounds (capping)
print(f"\n\n🔧 Capping outliers bằng IQR method...")

df_capped = df_clean.copy()

for col in audio_features:
    Q1 = df_capped[col].quantile(0.25)
    Q3 = df_capped[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap values
    df_capped[col] = df_capped[col].clip(lower_bound, upper_bound)

print("✅ Capping hoàn tất")

# Không xóa outliers, chỉ cap
df_clean = df_capped

print(f"\n📊 Shape sau xử lý outliers: {df_clean.shape}")

# ---------------------------------------------------------
#  ĐIỀN CÁC Ô NaN CÒN SÓT LẠI BẰNG NỘI SUY (INTERPOLATION)
# ---------------------------------------------------------
print(f"\n[ ĐIỀN NaN BẰNG NỘI SUY VÀ 'UNKNOWN']")

# Đếm NaN trước khi điền
nan_before_fill = df_clean.isnull().sum().sum()
print(f"   Tổng số ô trống trước khi điền: {nan_before_fill:,}")

if nan_before_fill > 0:
    # Hiển thị các cột còn có NaN
    print(f"\n   Các cột còn có NaN:")
    nan_cols_detail = {}
    for col in df_clean.columns:
        nan_count = df_clean[col].isnull().sum()
        if nan_count > 0:
            nan_cols_detail[col] = nan_count
            print(f"     • '{col}': {nan_count} NaN")
    
    # ĐIỀN BẰNG NỘI SUY (LINEAR INTERPOLATION) CHO CÁC CỘT SỐ
    print(f"\n   Áp dụng Linear Interpolation cho các cột số...")
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col] = df_clean[col].interpolate(method='linear', limit_direction='both')
            remaining_nan = df_clean[col].isnull().sum()
            print(f"     * '{col}': nội suy hoàn tất (NaN còn lại: {remaining_nan})")
    
    # ĐIỀN BẰNG 'UNKNOWN' CHO CÁC CỘT CHUỖI
    print(f"\n   Điền 'UNKNOWN' cho các cột chuỗi...")
    string_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
    for col in string_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col] = df_clean[col].fillna('UNKNOWN')
            print(f"     * '{col}': điền với 'UNKNOWN'")
    
    nan_after_fill = df_clean.isnull().sum().sum()
    print(f"\n   ✅ Đã điền xong! Tổng số ô trống sau khi điền: {nan_after_fill:,}")
else:
    print(f"   ✅ Không có ô NaN nào cần điền!")

print(f"\n[KẾT QUẢ ]")
print(f"   Shape: {df_clean.shape}")
print(f"   Tổng số ô trống (NaN): {df_clean.isnull().sum().sum():,}")


## 5. Chuẩn Hóa Kiểu Dữ Liệu (Data Type Standardization)

In [ ]:
print("="*80)
print("📝 CHUẨN HÓA KIỂU DỮ LIỆU")
print("="*80)

print("\n🔍 Kiểu dữ liệu hiện tại:")
print(df_clean.dtypes)

print("\n🔧 Chuẩn hóa kiểu dữ liệu...")

# LƯU Ý: Không dùng pd.to_numeric(..., errors='coerce') 
# vì nó chuyển 'UNKNOWN' strings thành NaN!

# Kiểm tra các cột số đã đúng dtype chưa
print("\nXác minh các kiểu dữ liệu:")
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
print(f"  Cột số: {len(numeric_cols)}")

string_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"  Cột chuỗi: {len(string_cols)}")
print(f"    Ví dụ: {string_cols[:5]}")

print(f"\n✅ Kiểu dữ liệu đã chính xác! (Cột số giữ nguyên kiểu số, chuỗi có 'UNKNOWN' giữ nguyên object)")
print(f"\n📝 Kiểu dữ liệu sau chuẩn hóa:")
print(df_clean.dtypes)


## 6. Chuẩn Hóa và Scale Các Features (Normalization & Scaling)

In [ ]:
print("="*80)
print("📊 CHUẨN HÓA VÀ SCALE AUDIO FEATURES")
print("="*80)

# Các audio features cần scale
audio_features_to_scale = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]
audio_features_to_scale = [f for f in audio_features_to_scale if f in df_clean.columns]

print(f"\n📈 Các features sẽ được scale: {audio_features_to_scale}")

# Hiển thị thống kê trước scaling
print(f"\n\n📊 Thống kê TRƯỚC scaling:")
print(df_clean[audio_features_to_scale].describe())

# Áp dụng MinMaxScaler (scale về [0, 1])
print(f"\n\n🔧 Áp dụng MinMaxScaler (scale về [0, 1])...")

scaler = MinMaxScaler()
df_scaled = df_clean.copy()
df_scaled[audio_features_to_scale] = scaler.fit_transform(df_clean[audio_features_to_scale])

print("✅ Scaling hoàn tất")

# Hiển thị thống kê sau scaling
print(f"\n📊 Thống kê SAU scaling:")
print(df_scaled[audio_features_to_scale].describe())

# Visualize trước/sau scaling
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('So Sánh Trước/Sau Scaling', fontsize=16, fontweight='bold')

features_to_plot = audio_features_to_scale[:6]

for idx, col in enumerate(features_to_plot):
    row = idx // 3
    col_idx = idx % 3
    
    ax = axes[row, col_idx]
    
    ax.hist(df_clean[col], bins=30, alpha=0.5, label='Trước', color='red', edgecolor='black')
    ax.hist(df_scaled[col], bins=30, alpha=0.5, label='Sau', color='blue', edgecolor='black')
    
    ax.set_title(f'{col}')
    ax.set_xlabel('Giá trị')
    ax.set_ylabel('Tần số')
    ax.legend()

plt.tight_layout()
plt.show()

print("\n✅ Visualization hoàn tất")


## 7. Tóm Tắt Kết Quả Làm Sạch Dữ Liệu

In [ ]:
print("\n" + "="*80)
print("💾 LƯU DỮ LIỆU ĐÃ LÀM SẠCH VÀ SCALE")
print("="*80)

import os

# Lưu dữ liệu đã làm sạch
output_path_clean = '../data/processed/spotify_cleaned_v2.csv'
print(f"\n[1] Lưu dữ liệu đã làm sạch...")
print(f"    Đường dẫn: {output_path_clean}")

# Xóa file cũ để tránh vấn đề cache
if os.path.exists(output_path_clean):
    os.remove(output_path_clean)
    print(f"    ✅ File cũ đã xóa")

# Lưu file mới
df_clean.to_csv(output_path_clean, index=False)
print(f"    ✅ File đã được ghi (NaN trong bộ nhớ: {df_clean.isnull().sum().sum()})")

# Xác minh file trên disk
df_verify_clean = pd.read_csv(output_path_clean)
nan_on_disk = df_verify_clean.isnull().sum().sum()
print(f"    ✓ XÁC MINH: NaN trên disk = {nan_on_disk}")
if nan_on_disk == 0:
    print(f"    ✅ THÀNH CÔNG: spotify_cleaned_v2.csv đã lưu với 0 NaN!")
else:
    print(f"    ⚠️ CẢNH BÁO: Vẫn còn NaN trên disk!")

# Lưu dữ liệu đã scale
output_path_scaled = '../data/processed/spotify_cleaned_scaled_v3.csv'
print(f"\n[2] Lưu dữ liệu đã scale...")
print(f"    Đường dẫn: {output_path_scaled}")

# Xóa file cũ
if os.path.exists(output_path_scaled):
    os.remove(output_path_scaled)
    print(f"    ✅ File cũ đã xóa")

# Lưu file mới
df_scaled.to_csv(output_path_scaled, index=False)
print(f"    ✅ File đã được ghi (NaN trong bộ nhớ: {df_scaled.isnull().sum().sum()})")

# Xác minh file trên disk
df_verify_scaled = pd.read_csv(output_path_scaled)
nan_on_disk_scaled = df_verify_scaled.isnull().sum().sum()
print(f"    ✓ XÁC MINH: NaN trên disk = {nan_on_disk_scaled}")
if nan_on_disk_scaled == 0:
    print(f"    ✅ THÀNH CÔNG: spotify_cleaned_scaled_v3.csv đã lưu với 0 NaN!")
else:
    print(f"    ⚠️ CẢNH BÁO: Vẫn còn NaN trên disk!")

print("\n" + "="*80)
print("✅ HOÀN THÀNH QUY TRÌNH LÀM SẠCH DỮ LIỆU")
print("="*80)
